# Training Data: Sampling, Labelling, Imbalance

**Course:** [ML in Practice](https://ml-viz.vercel.app/courses/ml-in-practice/06-training-data)

This notebook is the hands-on counterpart to the lesson. We build an *imbalanced* 2D synthetic dataset (99% negatives, 1% positives), train logistic regression four different ways, and watch what each fix does to the precision / recall / F1 trade-off.

1. **Naive training on the natural distribution.** Accuracy is misleadingly high; recall on the minority class is poor.
2. **Class-weighted loss** with $w_c = N / (K \cdot n_c)$. Recall improves at the cost of slightly lower accuracy; F1 goes up.
3. **Random oversampling** of the minority class. Compare metrics and flag the overfitting risk by watching the train-vs-test gap.
4. **Threshold tuning.** Train on the natural distribution, sweep the decision threshold, plot the precision-recall curve.
5. **Stratified split sanity check.** Show that random `train_test_split` on a 1% minority rate occasionally puts zero minority examples in the test set; stratified splitting never does.

Self-contained: NumPy + matplotlib only. No sklearn, no pandas, no network, no API keys.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)

plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor':   '#1a1d27',
    'axes.edgecolor':   '#2a2d3a',
    'axes.labelcolor':  '#e2e8f0',
    'text.color':       '#e2e8f0',
    'xtick.color':      '#94a3b8',
    'ytick.color':      '#94a3b8',
    'grid.color':       '#2a2d3a',
    'grid.alpha':       0.5,
})

BRAND  = '#6366f1'
TEAL   = '#2dd4bf'
ROSE   = '#fb7185'
ORANGE = '#f97316'
YELLOW = '#facc15'
MUTED  = '#475569'

## 1. Build the imbalanced dataset

A 2D synthetic problem: two Gaussian blobs whose centres are close enough that the decision boundary is non-trivial. Class 0 has 9,900 points; class 1 has 100 points. The natural minority rate is 1%.

We split off a *stratified* test set first — both train and test see the same 1% minority rate — so every metric we report below is on the same population the natural rate would produce in production.

In [ ]:
rng = np.random.default_rng(0)

N_MAJ = 9_900
N_MIN = 100

X_maj = rng.normal(loc=[-0.4,  0.0], scale=0.9, size=(N_MAJ, 2))
X_min = rng.normal(loc=[ 1.4,  0.6], scale=0.6, size=(N_MIN, 2))

X = np.vstack([X_maj, X_min])
y = np.concatenate([np.zeros(N_MAJ, dtype=int), np.ones(N_MIN, dtype=int)])

print(f'Total examples : {len(y):,}')
print(f'Class 0 (maj)  : {int((y == 0).sum()):,} ({(y == 0).mean()*100:.2f}%)')
print(f'Class 1 (min)  : {int((y == 1).sum()):,} ({(y == 1).mean()*100:.2f}%)')


def stratified_split(X, y, test_frac=0.25, seed=0):
    '''Return (X_tr, y_tr, X_te, y_te) — stratified by class label.'''
    r = np.random.default_rng(seed)
    idx = np.arange(len(y))
    tr_idx, te_idx = [], []
    for cls in np.unique(y):
        cls_idx = idx[y == cls]
        r.shuffle(cls_idx)
        n_te = int(round(len(cls_idx) * test_frac))
        te_idx.append(cls_idx[:n_te])
        tr_idx.append(cls_idx[n_te:])
    tr_idx = np.concatenate(tr_idx); r.shuffle(tr_idx)
    te_idx = np.concatenate(te_idx); r.shuffle(te_idx)
    return X[tr_idx], y[tr_idx], X[te_idx], y[te_idx]


X_tr, y_tr, X_te, y_te = stratified_split(X, y, test_frac=0.25, seed=1)
print(f'\nTrain : {len(y_tr):,}  (minority {(y_tr == 1).sum():,} = {(y_tr == 1).mean()*100:.2f}%)')
print(f'Test  : {len(y_te):,}  (minority {(y_te == 1).sum():,} = {(y_te == 1).mean()*100:.2f}%)')

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5))
ax.scatter(X[y == 0, 0], X[y == 0, 1], s=4, c=BRAND, alpha=0.45, label=f'class 0 (n = {N_MAJ:,})')
ax.scatter(X[y == 1, 0], X[y == 1, 1], s=18, c=ROSE, edgecolor='#0f1117', linewidth=0.5, label=f'class 1 (n = {N_MIN:,})')
ax.set_xlabel('x_1'); ax.set_ylabel('x_2')
ax.set_title(f'Imbalanced 2D dataset — 1% minority rate')
ax.grid(True); ax.legend(frameon=False)
plt.tight_layout(); plt.show()

## 2. Logistic regression in NumPy + four metric helpers

A tiny gradient-descent logistic regression — no sklearn. We accept an optional `sample_weight` so the same training loop supports both naive training and class-weighted training without forking the code.

The four classification metrics we report are the standard ones for imbalanced data:

- **Accuracy.** Fraction of predictions correct. Misleads on imbalanced data.
- **Precision.** Of the predicted positives, what fraction were actually positive?
- **Recall.** Of the actual positives, what fraction did we catch?
- **F1.** Harmonic mean of precision and recall, sensitive to both.

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -50, 50)))


def train_logreg(X, y, sample_weight=None, lr=0.1, n_iters=4000, seed=0):
    '''Train a logistic regression with optional per-example weights.'''
    r = np.random.default_rng(seed)
    n, d = X.shape
    Xb = np.hstack([X, np.ones((n, 1))])     # bias column
    w = r.normal(0, 0.01, size=d + 1)
    if sample_weight is None:
        sample_weight = np.ones(n)
    sw = sample_weight / sample_weight.sum() * n     # normalise so mean weight = 1
    for _ in range(n_iters):
        p = sigmoid(Xb @ w)
        grad = (Xb * (sw * (p - y))[:, None]).mean(axis=0)
        w -= lr * grad
    return w


def predict_proba(w, X):
    Xb = np.hstack([X, np.ones((len(X), 1))])
    return sigmoid(Xb @ w)


def classification_report(y_true, y_pred):
    '''Accuracy, precision, recall, F1 — minority-class oriented (positive = 1).'''
    tp = int(((y_pred == 1) & (y_true == 1)).sum())
    fp = int(((y_pred == 1) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum())
    tn = int(((y_pred == 0) & (y_true == 0)).sum())
    acc  = (tp + tn) / max(tp + tn + fp + fn, 1)
    prec = tp / max(tp + fp, 1)
    rec  = tp / max(tp + fn, 1)
    f1   = 2 * prec * rec / max(prec + rec, 1e-9)
    return {'acc': acc, 'prec': prec, 'rec': rec, 'f1': f1,
            'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn}


print('Helpers ready: train_logreg(), predict_proba(), classification_report().')

## 3. Naive training on the natural distribution

First, train without any imbalance correction. The default 0.5 threshold turns probabilities into hard predictions. Watch what happens to recall.

In [ ]:
w_naive = train_logreg(X_tr, y_tr)

p_te = predict_proba(w_naive, X_te)
y_pred_naive = (p_te >= 0.5).astype(int)
m_naive = classification_report(y_te, y_pred_naive)

print(f"Naive training (no imbalance correction):")
for k in ('acc', 'prec', 'rec', 'f1'):
    print(f'  {k:<8s} = {m_naive[k]:.3f}')
print(f'  confusion: tp={m_naive["tp"]}  fp={m_naive["fp"]}  fn={m_naive["fn"]}  tn={m_naive["tn"]}')

Accuracy is excellent — but the model is almost entirely composed of correct *majority* predictions. Recall on the rare positive class is poor because the gradient signal during training was dominated by the 99% of examples that are negative. This is the canonical failure mode: a 99%-accurate model that is useless for the thing you actually care about.

## 4. Class-weighted loss

Multiply each example's loss by $w_{y_i} = N / (K \cdot n_{y_i})$ — the sklearn convention. Rare-class examples now contribute the same *total* gradient over an epoch as the abundant class. Nothing else about training changes.

In [ ]:
def class_weights(y, K=None):
    '''sklearn-style balanced class weights:  w_c = N / (K * n_c).'''
    classes, counts = np.unique(y, return_counts=True)
    K = len(classes) if K is None else K
    N = len(y)
    return {int(c): float(N / (K * n)) for c, n in zip(classes, counts)}


cw = class_weights(y_tr)
print(f'class weights : {cw}')

sw = np.array([cw[int(c)] for c in y_tr])
w_weighted = train_logreg(X_tr, y_tr, sample_weight=sw)

p_te = predict_proba(w_weighted, X_te)
y_pred_w = (p_te >= 0.5).astype(int)
m_weighted = classification_report(y_te, y_pred_w)

print(f'\nClass-weighted training:')
for k in ('acc', 'prec', 'rec', 'f1'):
    print(f'  {k:<8s} = {m_weighted[k]:.3f}')
print(f'  confusion: tp={m_weighted["tp"]}  fp={m_weighted["fp"]}  fn={m_weighted["fn"]}  tn={m_weighted["tn"]}')

Recall on the minority class jumps because the model now pays attention to misclassifying positives. Accuracy drops slightly — the model accepts more false positives in exchange for more true positives — but F1 rises. The right axis to evaluate on imbalanced data is always F1 (or PR-AUC), never accuracy.

## 5. Random oversampling of the minority class

An alternative to weighted loss: replicate minority examples until the training set is balanced. Mathematically equivalent for un-regularised cross-entropy, but with a real downside — every minority point is seen many times per epoch, so the model can overfit to *that specific point* rather than the underlying minority distribution. We flag the risk by reporting the train-vs-test gap.

In [ ]:
def oversample(X, y, seed=0):
    '''Replicate minority-class examples until each class count matches the max.'''
    r = np.random.default_rng(seed)
    classes, counts = np.unique(y, return_counts=True)
    target = counts.max()
    parts_X, parts_y = [], []
    for c, n in zip(classes, counts):
        idx = np.where(y == c)[0]
        if n == target:
            parts_X.append(X[idx]); parts_y.append(y[idx])
        else:
            extra = r.choice(idx, size=target - n, replace=True)
            parts_X.append(np.concatenate([X[idx], X[extra]]))
            parts_y.append(np.concatenate([y[idx], y[extra]]))
    Xr = np.concatenate(parts_X, axis=0)
    yr = np.concatenate(parts_y, axis=0)
    perm = r.permutation(len(yr))
    return Xr[perm], yr[perm]


X_tr_os, y_tr_os = oversample(X_tr, y_tr, seed=2)
print(f'oversampled train size : {len(y_tr_os):,}  (minority {(y_tr_os == 1).sum():,} = {(y_tr_os == 1).mean()*100:.1f}%)')

w_os = train_logreg(X_tr_os, y_tr_os)

# Test metrics on the *natural-rate* test set.
p_te = predict_proba(w_os, X_te)
y_pred_os = (p_te >= 0.5).astype(int)
m_os = classification_report(y_te, y_pred_os)

# Train metrics on the oversampled distribution (the model's training reality)
# and on the original natural train set (the production analog).
p_tr_os = predict_proba(w_os, X_tr_os)
y_pred_tr_os = (p_tr_os >= 0.5).astype(int)
m_train_on_os = classification_report(y_tr_os, y_pred_tr_os)

p_tr_nat = predict_proba(w_os, X_tr)
y_pred_tr_nat = (p_tr_nat >= 0.5).astype(int)
m_train_on_nat = classification_report(y_tr, y_pred_tr_nat)

print(f'\nOversampling — test metrics (natural 1% distribution):')
for k in ('acc', 'prec', 'rec', 'f1'):
    print(f'  {k:<8s} = {m_os[k]:.3f}')

print(f'\nTrain-vs-test F1 gap (overfitting flag):')
print(f'  F1 on oversampled-train  = {m_train_on_os["f1"]:.3f}')
print(f'  F1 on natural-train      = {m_train_on_nat["f1"]:.3f}')
print(f'  F1 on test               = {m_os["f1"]:.3f}')
print(f'  train-test gap           = {m_train_on_os["f1"] - m_os["f1"]:+.3f}')

On *this* problem oversampling produces very similar test metrics to the class-weighted loss — the two methods are mathematically related — but the train-vs-test F1 gap is the diagnostic to watch. On higher-capacity models (random forests, gradient-boosted trees, neural nets) the gap from oversampling can be large because the model memorises the duplicated minority points. The standard playbook: prefer class-weighted loss for linear / logistic models; treat oversampling and SMOTE as escalations only when reweighting alone is insufficient.

## 6. Threshold tuning + precision-recall curve

Go back to the *naively* trained model and instead of touching the training data, sweep the decision threshold from 0 to 1. For each threshold we get a (precision, recall) point; the curve is the family of operating points the model can hit. The headline summary is the area under it (PR-AUC). Threshold tuning costs *nothing* — no retraining, no extra data — and is the cheapest fix for imbalanced data.

In [ ]:
p_te = predict_proba(w_naive, X_te)

thresholds = np.linspace(0.01, 0.99, 99)
precisions, recalls, f1s = [], [], []
for t in thresholds:
    yp = (p_te >= t).astype(int)
    m = classification_report(y_te, yp)
    precisions.append(m['prec'])
    recalls.append(m['rec'])
    f1s.append(m['f1'])
precisions = np.array(precisions); recalls = np.array(recalls); f1s = np.array(f1s)

best_idx = int(np.argmax(f1s))
print(f'Best F1 = {f1s[best_idx]:.3f} at threshold = {thresholds[best_idx]:.2f}')
print(f'  precision at that threshold = {precisions[best_idx]:.3f}')
print(f'  recall    at that threshold = {recalls[best_idx]:.3f}')
print(f'\nDefault 0.5 threshold:')
print(f'  F1 = {m_naive["f1"]:.3f}  prec = {m_naive["prec"]:.3f}  rec = {m_naive["rec"]:.3f}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

# Precision-recall curve.
order = np.argsort(recalls)
ax1.plot(recalls[order], precisions[order], color=BRAND, linewidth=2)
ax1.scatter([m_naive['rec']], [m_naive['prec']], s=70, color=ROSE, zorder=5, label='default threshold (0.5)')
ax1.scatter([recalls[best_idx]], [precisions[best_idx]], s=70, color=TEAL, zorder=5, label=f'best F1 threshold ({thresholds[best_idx]:.2f})')
ax1.set_xlabel('recall'); ax1.set_ylabel('precision')
ax1.set_title('Precision-Recall curve — naive model')
ax1.set_xlim(0, 1); ax1.set_ylim(0, 1.05)
ax1.grid(True); ax1.legend(frameon=False, loc='lower left')

# F1 vs threshold sweep.
ax2.plot(thresholds, f1s, color=ORANGE, linewidth=2)
ax2.axvline(thresholds[best_idx], color=TEAL, linestyle='--', label=f'best F1 at t = {thresholds[best_idx]:.2f}')
ax2.axvline(0.5, color=ROSE, linestyle='--', label='default t = 0.5')
ax2.set_xlabel('decision threshold'); ax2.set_ylabel('F1 on test')
ax2.set_title('Threshold sweep')
ax2.grid(True); ax2.legend(frameon=False)

plt.tight_layout(); plt.show()

Lowering the threshold below 0.5 trades precision for recall and lifts F1 substantially over the default-threshold naive model — without any change to training. Threshold tuning is the first fix to try on imbalanced data because it costs nothing; the model is already trained, you just pick the operating point on the curve that matches your application's precision/recall priorities.

## 7. Stratified-split sanity check

With a 1% positive rate, a random `train_test_split(..., test_size=0.25)` *can* produce a test set with zero or single-digit minority examples — at which point the reported recall is undefined or unstable across seeds. Stratified splitting fixes the same minority rate in train and test by construction. We sweep across 200 seeds and count how often a random split produces a degenerate test set.

In [ ]:
def random_split(X, y, test_frac=0.25, seed=0):
    r = np.random.default_rng(seed)
    perm = r.permutation(len(y))
    n_te = int(round(len(y) * test_frac))
    te = perm[:n_te]; tr = perm[n_te:]
    return X[tr], y[tr], X[te], y[te]


N_SEEDS = 200
rand_min_counts = []
strat_min_counts = []
for s in range(N_SEEDS):
    _, _, _, yte_r = random_split(X, y, test_frac=0.25, seed=s)
    _, _, _, yte_s = stratified_split(X, y, test_frac=0.25, seed=s)
    rand_min_counts.append(int((yte_r == 1).sum()))
    strat_min_counts.append(int((yte_s == 1).sum()))

rand_min_counts = np.array(rand_min_counts)
strat_min_counts = np.array(strat_min_counts)

print(f'Across {N_SEEDS} seeds:')
print(f'  random split — min/median/max minority count in test : {rand_min_counts.min()} / {int(np.median(rand_min_counts))} / {rand_min_counts.max()}')
print(f'  stratified  — min/median/max minority count in test : {strat_min_counts.min()} / {int(np.median(strat_min_counts))} / {strat_min_counts.max()}')
print(f'\n  random split with <5 minority in test : {int((rand_min_counts < 5).sum())} / {N_SEEDS} seeds')
print(f'  stratified    with <5 minority in test : {int((strat_min_counts < 5).sum())} / {N_SEEDS} seeds')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
bins = np.arange(min(rand_min_counts.min(), strat_min_counts.min()),
                  max(rand_min_counts.max(), strat_min_counts.max()) + 2)
ax.hist(rand_min_counts, bins=bins, alpha=0.65, color=ROSE, label='random split', edgecolor='#0f1117')
ax.hist(strat_min_counts, bins=bins, alpha=0.85, color=TEAL, label='stratified split', edgecolor='#0f1117')
ax.set_xlabel('minority count in test set')
ax.set_ylabel(f'seeds (out of {N_SEEDS})')
ax.set_title('Random vs stratified split: variance of test-set minority count')
ax.grid(True, axis='y')
ax.legend(frameon=False)
plt.tight_layout(); plt.show()

Random splitting on a 1% positive rate has a wide variance in the test set's minority count — some seeds land far below the expected 25 positives. Stratified splitting produces the same count every seed. The practical consequence: every metric you report on a randomly-split test set has an implicit confidence interval driven by the seed, and that confidence interval is large for the minority class. Stratified splitting eliminates the source of variance and is one line of code; there is no excuse to skip it on imbalanced data.

## Summary table

Final F1 / precision / recall for each strategy, all evaluated on the same stratified, natural-rate test set.

In [ ]:
p_te_naive = predict_proba(w_naive, X_te)
best_t = thresholds[best_idx]
y_pred_tuned = (p_te_naive >= best_t).astype(int)
m_tuned = classification_report(y_te, y_pred_tuned)

rows = [
    ('naive (t = 0.5)',                   m_naive),
    ('class-weighted loss (t = 0.5)',     m_weighted),
    ('oversampling (t = 0.5)',            m_os),
    (f'naive + threshold (t = {best_t:.2f})', m_tuned),
]

print(f'{"strategy":<32s} {"acc":>6s} {"prec":>6s} {"rec":>6s} {"F1":>6s}')
print('-' * 60)
for name, m in rows:
    print(f'{name:<32s} {m["acc"]:>6.3f} {m["prec"]:>6.3f} {m["rec"]:>6.3f} {m["f1"]:>6.3f}')

---
## ✏️ Your turn

### Exercise: implement `class_weights(y)`

Given a 1-D NumPy array `y` of integer class labels, return a `dict` mapping each unique class label `c` to its sklearn-style balanced weight

$$w_c = \frac{N}{K \cdot n_c}$$

where $N$ is the total number of examples, $K$ is the number of distinct classes, and $n_c$ is the count of class $c$.

**Reference value.** For a 90/10 dataset with 900 examples of class 0 and 100 examples of class 1, $K = 2$ and $N = 1000$, so $w_0 = 1000 / (2 \cdot 900) \approx 0.556$ and $w_1 = 1000 / (2 \cdot 100) = 5.0$.

In [ ]:
def class_weights_yours(y):
    '''Return {class_label: balanced_weight} using w_c = N / (K * n_c).

    Args:
        y : 1-D NumPy array of integer class labels.

    Returns:
        dict mapping each class label to its weight.
    '''
    # TODO(you):
    #   1. Use np.unique(y, return_counts=True) to get classes and counts.
    #   2. N = len(y), K = number of classes.
    #   3. weight_c = N / (K * count_c) for each class.
    #   4. Return as dict {int(c): float(w_c)}.
    pass


# Smoke run.
y_demo = np.concatenate([np.zeros(900, dtype=int), np.ones(100, dtype=int)])
print(f'class_weights_yours(y_demo) = {class_weights_yours(y_demo)}')

In [ ]:
# Test 1: 90/10 dataset.
y_test = np.concatenate([np.zeros(900, dtype=int), np.ones(100, dtype=int)])
w = class_weights_yours(y_test)
assert w is not None, 'function returned None'
assert abs(w[0] - 1000 / (2 * 900)) < 1e-9, f'w[0] = {w[0]}'
assert abs(w[1] - 1000 / (2 * 100)) < 1e-9, f'w[1] = {w[1]}'

# Test 2: 3-class problem.
y_test = np.concatenate([np.zeros(900, dtype=int),
                          np.ones(80,  dtype=int),
                          np.full(20, 2, dtype=int)])
w = class_weights_yours(y_test)
N, K = 1000, 3
assert abs(w[0] - N / (K * 900)) < 1e-9
assert abs(w[1] - N / (K * 80))  < 1e-9
assert abs(w[2] - N / (K * 20))  < 1e-9
# Rare class should have the largest weight.
assert w[2] > w[1] > w[0]

# Test 3: balanced data -> every weight equals 1.
y_test = np.array([0, 0, 1, 1, 2, 2])
w = class_weights_yours(y_test)
for c in (0, 1, 2):
    assert abs(w[c] - 1.0) < 1e-9, f'balanced data: w[{c}] = {w[c]}'

print('All class-weight tests passed.')

<details>
<summary>Show solution</summary>

```python
def class_weights_yours(y):
    classes, counts = np.unique(y, return_counts=True)
    N, K = len(y), len(classes)
    return {int(c): float(N / (K * n)) for c, n in zip(classes, counts)}
```

Three things to notice:

1. **Balanced data → unit weights.** When every class has the same count, $N / (K \cdot n_c) = 1$ for every $c$, so the weighted loss collapses back to the unweighted one. The weights only do work when the class counts differ.
2. **The weights sum to a constant.** $\sum_c n_c \cdot w_c = \sum_c n_c \cdot N / (K \cdot n_c) = \sum_c N / K = N$ — the *total* sample weight equals the number of examples, which is why the weighted loss is on the same scale as the unweighted one and the same learning rate usually works.
3. **You can use this for the loss or for sample weights.** Modern training frameworks accept *either* a `class_weight` dict (one number per class) or a `sample_weight` tensor (one number per example). They are equivalent under the hood: `sample_weight[i] = class_weight[y[i]]`. Pick whichever your library wants.
</details>